In [1]:
import datetime
import keras
import os
import re
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.feature_extraction.text import CountVectorizer
from time import time
from tmu.models.autoencoder.autoencoder import TMAutoEncoder
from contextlib import redirect_stdout
from tqdm import tqdm
from nltk.tokenize import word_tokenize
from sklearn.datasets import fetch_20newsgroups


# target_words = ['london', 'city', 'lousy', 'abysmal', 'crap', 'terrible', 'brilliant', 'excellent', 'superb', 'magnificent', 'marvellous', 'truck', 'plane', 'car', 'cars', 'motorcycle',  'scary', 'frightening', 'terrifying', 'horrifying', 'funny', 'comic', 'hilarious', 'witty']
# 'yogurt',
# target_words = [
#     'apple', 'book', 'cat', 'dog', 'elephant', 'friend', 'house', 'internet',
#     'journey', 'language', 'machine', 'night', 'orange', 'person', 'queen',
#     'river', 'tree', 'university', 'village', 'water', 'resilience', 'kindness',
#     'ball', 'car', 'desk', 'elephant', 'flower', 'guitar', 'hat', 'island',
#     'jacket', 'king', 'laptop', 'mouse', 'notebook', 'orange', 'pillow', 'sun',
#     'rainbow', 'sun', 'table', 'umbrella', 'van', 'wallet', 'queen','frog',
#     'zebra', 'apple', 'banana', 'cat', 'dog', 'elephant',  'giraffe', 'violin', 
#     'horse', 'kiwi', 'lion', 'monkey', 'nut', 'orange','youth','unity','quiet',
#     'quail', 'rabbit', 'strawberry', 'tiger', 'umbrella', 'victory','silence',
#     'yak', 'zebra', 'art', 'beauty', 'creativity', 'zest','talent', 'wisdom',
#     'dance', 'education', 'fashion', 'gardening', 'happiness', 'imagination',
#     'joy', 'laughter', 'music', 'nature', 'optimism', 'passion','zoo','year'
# ]
target_words = [
    'apple','orange'
]
home_dir = os.path.expanduser("~")
root_folder = os.path.join(home_dir, "tmu_results")
if not os.path.exists(root_folder):
    os.makedirs(root_folder)


2023-12-19 17:25:46,623 - tmu.util.cuda_profiler - WARNING - Could not import pycuda: No module named 'pycuda'
2023-12-19 17:25:46,627 - tmu.clause_bank.clause_bank_cuda - ERROR - No module named 'pycuda'
Traceback (most recent call last):
  File "C:\Users\ahmedkk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tmu\clause_bank\clause_bank_cuda.py", line 41, in <module>
    from pycuda._driver import Device, Context
ModuleNotFoundError: No module named 'pycuda'
2023-12-19 17:25:46,628 - tmu.clause_bank.clause_bank_cuda - WARNING - Could not import pycuda. This indicates that it is not installed! A possible fix is to run 'pip install pycuda'. Fallback to CPU ClauseBanks.


In [2]:
clause_weight_threshold = 0
number_of_examples = 4000
factor = 4
T = factor*40
s = 5.0
#clauses = factor*5
clauses = 40
clause_increment = False
random_per_category = False
categories = 0
accumulation = 24
epochs = 2

In [5]:
experts_dataset = []

def prepare_bbc():
    # import nltk
    # nltk.download('punkt')
    data_dir ="bbc"
    no_docs = 0
    for category in os.listdir(data_dir):
        category_dir = os.path.join(data_dir, category)
        no_docs = no_docs + len(os.listdir(category_dir))
        for file_name in os.listdir(category_dir):
            file_path = os.path.join(category_dir, file_name)
            with open(file_path, "r") as file:
                content = file.read()
                preprocessed_content = re.sub(r'\W+', ' ', content.lower())
                tokens = word_tokenize(preprocessed_content)
                experts_dataset.append(tokens)
    print("the number of news in train set =",no_docs)

def prepare_20newsgroups():
    newsgroups_data = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
    documents = newsgroups_data.data
    print("the number of news in train set =",len(documents))
    for document in documents:
        preprocessed_content = re.sub(r'\W+', ' ', document.lower()).split()
        experts_dataset.append(preprocessed_content)

def prepare_ag_news():
    # pip install tensorflow_datasets
    # pip install datasets
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    import tensorflow_datasets as tfds
    # data = tfds.load('ag_news', split='train[:25000]')
    data = tfds.load('ag_news', split='train')
    print("the number of news in train set =",len(data))
    for sample in data:
        experts_dataset.append(sample['text'].numpy().decode('utf-8').split())

def prepare_imdb():
    NUM_WORDS=10000
    INDEX_FROM=2
    train,test = keras.datasets.imdb.load_data(num_words=NUM_WORDS, index_from=INDEX_FROM)
    train_x,train_y = train
    print("the number of reviews in train set =",train_x.size)
    word_to_id = keras.datasets.imdb.get_word_index()
    word_to_id = {k:(v+INDEX_FROM) for k,v in word_to_id.items()}
    word_to_id.pop("<PAD>", None)
    word_to_id.pop("<START>", None)
    word_to_id.pop("<UNK>", None)
    id_to_word = {value:key for key,value in word_to_id.items()}
    for i in range(train_y.shape[0]):
        terms = []
        for word_id in train_x[i]:
            if word_id in id_to_word:
                terms.append(id_to_word[word_id].lower())             
        experts_dataset.append(terms)

In [ ]:
combined = True
experts_dataset = []
involved_datasets = []

if combined == True:
    involved_datasets.append(["Combined",0,1])

# print("start preparing BBC Sport datatsets")
# old_length = len(experts_dataset)
# prepare_bbc()
# if combined == False:
#     involved_datasets.append(["BBC",old_length,len(experts_dataset)])

# print("start preparing 20 Newsgroups datatsets")
# old_length = len(experts_dataset)
# prepare_20newsgroups()
# if combined == False:
#     involved_datasets.append(["20 News",old_length,len(experts_dataset)])

# print("start preparing AG News datatsets")
# old_length = len(experts_dataset)
# prepare_ag_news()
# if combined == False:
#     involved_datasets.append(["AG News",old_length,len(experts_dataset)])

print("start preparing IMDB datatsets")
old_length = len(experts_dataset)
prepare_imdb()
if combined == False:
    involved_datasets.append(["IMDB",old_length,len(experts_dataset)])

print("datatsets completed")
print(involved_datasets)

In [ ]:
def tokenizer(s):
    return s
vectorizer_X = CountVectorizer(tokenizer=tokenizer, lowercase=False, binary = True)
X_train = vectorizer_X.fit_transform(experts_dataset)
feature_names = vectorizer_X.get_feature_names_out()
number_of_features = vectorizer_X.get_feature_names_out().shape[0]
print("No of features: %d" % number_of_features)
output_active = np.empty(len(target_words), dtype=np.uint32)
for i in range(len(target_words)):
    target_word = target_words[i]
    if target_word in vectorizer_X.vocabulary_:
        target_id = vectorizer_X.vocabulary_[target_word]
        output_active[i] = target_id
    else:
        print(f"Warning: '{target_word}' not found in vocabulary.")
print("tokenizing target words completed")

In [ ]:
# london was 58033
# target_id = vectorizer_X.vocabulary_["london"]
# target_id = 75192
# print(vectorizer_X.get_feature_names_out()[target_id])
# print(experts_dataset[220])
# print(experts_dataset[1401])


# terms = []
# data_list = X_train[involved_datasets[0][1] - 1]
# original_document = vectorizer_X.inverse_transform(data_list)

# print("Original document:", original_document)

In [ ]:
num_segments = len(target_words)
step_size = int(360 / num_segments)
color_dict = {}
hue = 0
for word in target_words:
    color = plt.get_cmap('hsv')(hue / 360.0)
    color = mcolors.to_rgb(color)
    hex_color = mcolors.rgb2hex(color)
    color_dict[word] = hex_color
    hue += step_size

In [ ]:
file_path = "result/output.txt"
if os.path.exists(file_path):
    os.remove(file_path)
current_time = datetime.datetime.now()
test_id = current_time.strftime("%Y%m%d%H%M%S")
result_filename = f"result_{test_id}.txt"
test_dir = os.path.join(root_folder, test_id)
os.makedirs(test_dir)
result_filepath = os.path.join(test_dir, result_filename)
clauses_dir = os.path.join(test_dir, 'clauses_variance_plots')
if not os.path.exists(clauses_dir):
    os.makedirs(clauses_dir)

# parameters
clause_weight_threshold = 0
number_of_examples = 4000
factor = 4
T = factor*40
s = 5.0
#clauses = factor*5
clauses = 40
clause_increment = False
random_per_category = False
categories = 0
accumulation = 24
epochs = 2
fluctuations = []
progress_bar = tqdm(total=epochs, desc="Running Epochs")
# combined
target_words_clauses = []
    
with open(result_filepath, 'w') as file, redirect_stdout(file):
    print("Test: %s" % test_id)

    class_index = np.arange(len(output_active), dtype=np.uint32)
    for i in output_active:
        target_word_clauses = []

        shape = (1, 1)
        single_output_active = np.empty(1, dtype=np.uint32)
        single_output_active[0] = i
        tm = TMAutoEncoder(clauses, T, s, single_output_active, max_included_literals=3, accumulation=accumulation, feature_negation=False, platform='CPU', output_balancing=True)
        total_training = 0
        if(categories > 0): 
            print("Algorithm: With %d categories and random per category" % categories) if(random_per_category) else print("Algorithm: With %d categories" % categories) 
        else: 
            print("Algorithm: Original without categories")
        print("Epochs: %d" % epochs)
        print("Example: %d" % number_of_examples)
        print("Target words: %d" % len(target_words))
        print("Accumulation: %d" % accumulation)
        print("Datasets involved: %s" % involved_datasets)
        print("No of features: %d" % number_of_features)
        print("Clauses: %d with increment by 2\n" % clauses) if clause_increment else print("Clauses: %d\n" % clauses)
        
        for e in range(epochs):
            print("\nEpoch #%d" % (e+1))
            start_training = time()
            if categories > 0:
                tm.fit(
                    X_train, 
                    number_of_examples=number_of_examples, 
                    categories=categories, 
                    random_per_category = random_per_category,
                    involved_datasets=involved_datasets 
                    )
            else:
                tm.fit(
                    X_train, 
                    number_of_examples=number_of_examples, 
                    involved_datasets=involved_datasets 
                    )
            stop_training = time()
            total_training = total_training + (stop_training - start_training)

            
            if((e+1) == epochs):
                print("\n=====================================\nClauses\n=====================================")
                for j in range(clauses):
                    print("Clause #%-2d " % (j), end=' ')
                    l = [] 
                    related_literals = []
                    number_of_literals = 0 
                    for k in range(tm.clause_bank.number_of_literals):
                        if tm.get_ta_action(j, k) == 1:
                            number_of_literals = number_of_literals + 1
                            if k < tm.clause_bank.number_of_features:
                                related_literals.append(k)
                                l.append("%s(%d)" % (feature_names[k], tm.clause_bank.get_ta_state(j, k)))
                            else:
                                l.append("¬%s(%d)" % (feature_names[k-tm.clause_bank.number_of_features], tm.clause_bank.get_ta_state(j, k)))
                    print(": No of features:%-6d" % (number_of_literals), end=" ==> ")
                    try:
                        print(" - ".join(l).encode('utf-8', errors='ignore'))
                    except UnicodeEncodeError:
                        print(" exception ")
                    target_word_clauses.append([related_literals])
                print(target_word_clauses)
        
            # profile = np.empty((len(target_words), clauses))
            # for i in range(len(target_words)):
            #     weights = tm.get_weights(i)
            #     profile[i,:] = np.where(weights >= clause_weight_threshold, weights, 0)
        
            # similarity = cosine_similarity(profile)

            
            # print("\n=====================================\nWord Similarity\n=====================================")
            # max_word_length = len(max(target_words, key=len))
            # list_of_words = []
            # target_words_with_min_max = []
            # fluctuations_per_epoch = []
            # for i in range(len(target_words)):
            #     row_of_similarity = []
            #     sorted_index = np.argsort(-1*similarity[i,:])
            #     min_similarity = 1.0
            #     max_similarity = 0.0
            #     word_similarity = []
            #     for j in range(1, len(target_words)):
            #         row_of_similarity.append(target_words[sorted_index[j]])
            #         word_similarity.append("{:<{}}({:.2f})  ".format(target_words[sorted_index[j]], max_word_length, similarity[i, sorted_index[j]]))
            #         if(min_similarity > similarity[i,sorted_index[j]]):
            #             min_similarity = similarity[i,sorted_index[j]]
            #         if(max_similarity < similarity[i,sorted_index[j]]):
            #             max_similarity = similarity[i,sorted_index[j]]
        
            #     fluctuations_per_epoch.append((target_words[i],min_similarity * 100,max_similarity * 100))
            #     output_line = f"{target_words[i]:<{max_word_length}}: Min:{min_similarity:.2f}, Max:{max_similarity:.2f}"
            #     print(output_line, end='     ==> ')
            #     print(word_similarity)
            #     list_of_words.append(row_of_similarity)
            #     target_words_with_min_max.append(output_line)
            # fluctuations.append(fluctuations_per_epoch)
            progress_bar.update(1)
        target_words_clauses.append([i,target_word_clauses])

    seconds = total_training
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    seconds = seconds % 60
    print(f"Training duration: {hours} hours, {minutes} minutes, {seconds} seconds")
progress_bar.close()

In [ ]:
print(target_words_clauses)
with open("saved_variables.pkl", "wb") as file:
    pickle.dump((output_active, feature_names, number_of_features,target_words_clauses), file)

In [3]:
file_path = "result/output.txt"
if os.path.exists(file_path):
    os.remove(file_path)

with open("saved_variables.pkl", "rb") as file:
    output_active, feature_names, number_of_features, target_words_clauses = pickle.load(file)
    print(target_words_clauses)


[[600, [[[1602, 7884, 9225, 9674]], [[75, 3768, 6736]], [[3616, 8604]], [[488, 952, 3941, 9056]], [[5933]], [[257, 3894, 5197]], [[668, 3898, 5420, 5499, 9244, 9377]], [[445, 717, 2601, 3846, 4814, 5329, 6141, 8946]], [[9604]], [[1865, 5603]], [[2168, 9657]], [[3078, 7066, 8306, 9674]], [[8904, 9670]], [[681, 902, 2490, 5821, 7470, 8089]], [[3885, 9056]], [[2702, 3135, 5599, 7762]], [[2168, 8949, 9243, 9804]], [[3550, 9962]], [[8806]], [[369, 6868]], [[1500, 2695]], [[2635, 5746, 7300]], [[2327, 5744, 6747]], [[486, 4468, 6329]], [[3579, 4725, 5378, 5512]], [[2728, 3191, 8256]], [[]], [[3264, 4190, 4325, 6650, 8756]], [[5273, 5373, 5454]], [[3579, 4497, 9024]], [[1881, 6807, 8949]], [[422, 682, 5111, 5371, 7185, 9686]], [[1162, 5539]], [[1097]], [[118, 624, 6981, 7589]], [[2134, 6141]], [[3616, 4582, 5238, 5239]], [[8966, 9452]], [[3090, 4150, 4298]], [[165, 2775, 5503, 9942]]]], [6306, [[[1783, 3327, 3372, 6868]], [[1107, 1757, 3289]], [[8434]], [[1783, 6336, 6867]], [[329, 1783]], [[

In [4]:
tm = TMAutoEncoder(clauses, T, s, output_active, max_included_literals=3, accumulation=accumulation, feature_negation=False, platform='CPU', output_balancing=True)
total_training = 0
print("Epochs: %d" % epochs)
print("Target words: %d" % len(target_words))
print("Accumulation: %d" % accumulation)
print("No of features: %d" % number_of_features)
print("Clauses: %d\n" % clauses)

for e in range(epochs):
    print("\nEpoch #%d" % (e+1))
    start_training = time()
    tm.fit_combined(
        number_of_examples = 100,
        target_words_clauses = target_words_clauses
        )
    stop_training = time()
    total_training = total_training + (stop_training - start_training)

    
    if((e+1) == epochs):
        print("\n=====================================\nClauses\n=====================================")
        for j in range(clauses):
            print("Clause #%-2d " % (j), end=' ')
            l = [] 
            number_of_literals = 0 
            for k in range(tm.clause_bank.number_of_literals):
                if tm.get_ta_action(j, k) == 1:
                    number_of_literals = number_of_literals + 1
                    if k < tm.clause_bank.number_of_features:
                        l.append("%s(%d)" % (feature_names[k], tm.clause_bank.get_ta_state(j, k)))
                    else:
                        l.append("¬%s(%d)" % (feature_names[k-tm.clause_bank.number_of_features], tm.clause_bank.get_ta_state(j, k)))
            print(": No of features:%-6d" % (number_of_literals), end=" ==> ")
            try:
                print(" ∧ ".join(l))
            except UnicodeEncodeError:
                print(" exception ")

Epochs: 2
Target words: 2
Accumulation: 24
No of features: 9997
Clauses: 40


Epoch #1
Starting the combined fitting...
Count of all related_literals: 243
Maximum feature: 9962
+------------+------------+------------+------------+------------+------------+------------+------------+
|   Column 1 |   Column 2 |   Column 3 |   Column 4 |   Column 5 |   Column 6 |   Column 7 |   Column 8 |
+============+============+============+============+============+============+============+============+
|       1602 |       7884 |       9225 |       9674 |          0 |          0 |          0 |          0 |
+------------+------------+------------+------------+------------+------------+------------+------------+
|         75 |       3768 |       6736 |          0 |          0 |          0 |          0 |          0 |
+------------+------------+------------+------------+------------+------------+------------+------------+
|       3616 |       8604 |          0 |          0 |          0 |          0 |  

In [ ]:
# drawing the clauuses variations
# color_array = np.zeros((len(list_of_words), len(list_of_words[0]), 3), dtype=np.uint8)
# for i in range(len(list_of_words)):
#     for j in range(len(list_of_words[i])):
#         word = list_of_words[i][j]
#         # Replace the word with its corresponding color
#         color = color_dict[word]
#         r, g, b = mcolors.hex2color(color)
#         color_array[i][j] = np.array([r*255, g*255, b*255], dtype=np.uint8)
#     # Add the row number in front of each row
#     plt.text(-0.5, i, f'{target_words_with_min_max[i]}', ha='right', va='center')
# # Create a plot of the color array
# plt.imshow(color_array)
# plt.axis('off')
# # Add legends for each word color
# legend_handles = []
# legend_labels = []
# for word, color in color_dict.items():
#     legend_handles.append(plt.Line2D([0], [0], marker='o', color='w', label=word, markerfacecolor=color, markersize=10))
#     legend_labels.append(word)
# plt.legend(handles=legend_handles, labels=legend_labels, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4)
# plot_filename = os.path.join(clauses_dir,f'clauses_{clauses}_{e+1}.png')
# plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
# plt.show()
# end of ploting clauses variation

In [ ]:
# from sklearn.manifold import TSNE
# import matplotlib.pyplot as plt
# import logging

# # Set the log level to suppress debug messages from matplotlib.font_manager
# logging.getLogger("matplotlib.font_manager").setLevel(logging.INFO)

# # Increase the number of samples in the dataset or decrease the perplexity
# n_samples = similarity.shape[0]
# perplexity = min(30, n_samples - 1)  # You can adjust the perplexity as needed

# # Perform t-SNE with adjusted perplexity
# tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
# embeddings = tsne.fit_transform(similarity)

# # Plot the results
# plt.figure(figsize=(8, 8))
# for i, word in enumerate(target_words):
#     plt.scatter(embeddings[i, 0], embeddings[i, 1])
#     plt.text(embeddings[i, 0], embeddings[i, 1], word)

# plt.title('t-SNE Visualization of Cosine Similarities')
# result_tSNE_path = os.path.join(test_dir, f"tSNE_{test_id}.png")
# plt.savefig(result_tSNE_path, dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
# target_words_fluctuations = []
# target_word_indices = [target_words.index(word) for word in target_words]
# fluctuations_int = np.array(fluctuations.copy())
# for target_word_index in target_word_indices:
#     target_words_fluctuations.append([[float(item[1]), float(item[2])] for item in fluctuations_int[:, target_word_index]])


# num_rows = int(np.ceil(len(target_words) / 3))  # Number of rows in the grid (assumes 3 columns)
# fig, axes = plt.subplots(num_rows, 3, figsize=(20, 40), sharex=True, sharey=True)

# for i, ax in enumerate(axes.flat):
#     if i < len(target_words_fluctuations):
#         target_word_fluctuations = target_words_fluctuations[i]
#         for epoch, similarity_of_word in enumerate(target_word_fluctuations):
#             ax.vlines(epoch+1, similarity_of_word[0], similarity_of_word[1], color='blue', alpha=0.5)  # Set alpha to add transparency to lines

#         ax.set_title(target_words[i])
#         ax.grid(True)

# fig.delaxes(axes[-1, -1])  # Remove any unnecessary subplot

# fig.text(0.5, 0.04, 'Epoch', ha='center')  # Common x-axis label
# fig.text(0.04, 0.5, 'Similarity', va='center', rotation='vertical')  # Common y-axis label
# plt.tight_layout()  # Adjust spacing between subplots
# result_image_path = os.path.join(test_dir, f"fluctuations_{test_id}.png")
# plt.savefig(result_image_path, dpi=300, bbox_inches='tight')
# plt.show()